<a href="https://colab.research.google.com/github/yosie111/cadcam/blob/main/stage1_cadcam_v1_heb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# שלב 1 — CAD/CAM V1: המערכת הפרוצדורלית הישנה

**מקור:** *Design Patterns Explained* (Shalloway & Trott), פרקים 3 ו-12

---

## הבעיה במילים שלי

חברת הנדסה גדולה משתמשת ב**מערכת CAD/CAM** לעיצוב חלקי פח מתכת.  
כל חלק הוא יריעת מתכת עם צורות חתוכות בתוכה. לצורות האלה קוראים **features**  
והן מגיעות בחמישה סוגים: **slot, hole, cutout, special, irregular**.

קיימת כבר **מערכת מומחה** יקרה שיודעת את חוקי הייצור —  
באיזה סדר לחתוך, אילו כלים להשתמש וכו'. התפקיד שלנו הוא לכתוב  
**חולץ גיאומטריה** שקורא מודל CAD/CAM ומזין את ה-features למערכת המומחה.

הסיבוך: ספק ה-CAD/CAM עובר מ-**Version 1** (פרוצדורלי) ל-**Version 2** (מונחה עצמים),  
ואנחנו חייבים לתמוך בשניהם בלי לשנות את מערכת המומחה.

### השלב הזה: V1 בלבד, פרוצדורלי לחלוטין

V1 הוא אוסף של **ספריות פונקציות בסגנון C**. שליפת מידע דורשת:

1. פתיחת מודל לפי שם ← קבלת **handle** (מספר שלם אטום).
2. שאלה כמה features יש במודל ← קבלת **מספר**.
3. לולאה מ-0 עד count. לכל אינדקס — בקשת **ID** ו**סוג** של ה-feature.
4. לפי הסוג — קריאה ל**getter הנכון לפי הסוג** (למשל `v1_get_x_for_slot`).

הקורא חייב **לנהל ידנית** את ה-handle, את אינדקס הלולאה, ואת ה-dispatch לפי סוג.  
הספר מתאר את זה כ-*"painful to deal with and clearly not object-oriented."*

### הנחות

- אנחנו מדמים את ספריית V1 כפונקציות Python עם dict בזיכרון.
- ערכי הגיאומטריה (x, y, width וכו') הם floats פשוטים לצורך בהירות.
- מערכת המומחה היא stub קטן — רק מקבלת dicts ומדפיסה פקודות NC.
- אנחנו בכוונה משאירים הכל פרוצדורלי (בלי מחלקות ל-features) כדי להרגיש את הכאב.

---

## זרימת הקריאות

```text
extract_features_v1("PART-001")
  │
  ├── v1_open_model("PART-001")          → handle = 1
  │
  ├── v1_get_num_features(handle)        → count = 9
  │
  ├── for i in range(count):
  │     │
  │     ├── v1_get_feature_id(handle, i)     → fid = ...
  │     ├── v1_get_feature_type(handle, i)   → ftype = "slot" / "hole" / ...
  │     │
  │     └── if ftype == "slot":
  │           ├── v1_get_x_for_slot(handle, fid)      ← תלוי סוג!
  │           ├── v1_get_y_for_slot(handle, fid)      ← תלוי סוג!
  │           ├── v1_get_w_for_slot(handle, fid)      ← תלוי סוג!
  │           └── ...                                  ← סט שונה לכל סוג
  │
  │         elif ftype == "hole":
  │           ├── v1_get_x_for_hole(handle, fid)      ← פונקציה אחרת!
  │           ├── v1_get_radius_for_hole(handle, fid)  ← ייחודי ל-hole!
  │           └── ...
  │
  └── v1_close_model(handle)             ← אסור לשכוח!
```

**הכאב המרכזי:** כל סוג feature דורש קריאה ל**פונקציות שונות**.  
קריאה לפונקציה הלא-נכונה על הסוג הלא-נכון = באג שקט (או קריסה ב-C).

---

## איזה getters שייכים לאיזה סוג?

ל-V1 **אין ממשק אחיד**. לכל סוג יש סט getters משלו:

| סוג Feature  | Getters בשימוש                            | כמות |
|:-------------|:------------------------------------------|------:|
| slot         | x, y, width, height, angle, depth         | 6     |
| hole         | x, y, radius, depth                       | 4     |
| cutout       | x, y, width, height, angle                | 5     |
| special      | x, y, code                                | 3     |
| irregular    | x, y, bbox_w, bbox_h                      | 4     |

שימו לב: `x` ו-`y` מופיעים בכל סוג, אבל V1 עדיין מכריח אותנו לקרוא  
`v1_get_x_for_slot` מול `v1_get_x_for_hole` — **פונקציות שונות** שעושות אותו דבר.  
זה הכאב המרכזי של V1.

---

## ארכיטקטורה

```
       ┌───────────────┐
       │   מסד נתונים   │   (מדמה את מסד הנתונים של V1)
       │   V1 בזיכרון   │
       └───────┬───────┘
               │
    ┌──────────┴──────────┐
    │  ספריית הפונקציות   │   חלקים B-E: v1_open_model,
    │  של V1              │   v1_get_x_for_slot, ...
    └───────┬──────────┘
            │
    ┌───────┴──────────┐
    │  לוגיקת החילוץ   │   חלק G: ניהול handle,
    │  (פרוצדורלי)      │   לולאה, if/elif dispatch
    └───────┬──────────┘
            │  list[dict]
    ┌───────┴──────────┐
    │  מערכת המומחה    │   חלק H: מקבלת dicts,
    │  (stub)          │   מייצרת פקודות NC
    └──────────────────┘
```

הכל פונקציות ו-dicts. בלי מחלקות, בלי ירושה, בלי design patterns.

---
## חלק A: מסד נתונים גולמי + קבועים לשמות שדות
ה-V1 האמיתי שומר נתונים בפורמט בינארי קנייני.  
אנחנו מדמים אותו עם dict מקונן. כל רשומת feature היא **tuple שטוח** (כמו struct ב-C).  
קבועים עם שמות הופכים את מיקומי ה-tuple לקריאים — עדיין פרוצדורלי, אבל ברור.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק A: מסד נתונים גולמי + קבועים לשמות שדות
# ────────────────────────────────────────────────────────────

from __future__ import annotations

# קבועים למיקומים ב-tuple (במקום מספרי קסם כמו [2], [3])
FID          = 0   # מזהה feature
FTYPE        = 1   # מחרוזת סוג: "slot", "hole", ...
X            = 2   # קואורדינטת x
Y            = 3   # קואורדינטת y
WIDTH        = 4   # רוחב (או רוחב bbox ל-irregular)
HEIGHT       = 5   # גובה (או גובה bbox ל-irregular)
ANGLE        = 6   # זווית סיבוב במעלות
DEPTH        = 7   # עומק חיתוך
RADIUS       = 8   # רדיוס חור (None לסוגים אחרים)
SPECIAL_CODE = 9   # קוד פאנץ' מיוחד (None לסוגים אחרים)

_V1_DATABASE: dict[str, dict] = {
    "PART-001": {
        "sheet_width": 200.0,
        "sheet_height": 150.0,
        "material": "Aluminum-6061",
        "features": [
            #  FID  FTYPE        X      Y      W     H    ANG  DEP   RAD    SCODE
            (   1, "slot",     10.0,  30.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            (   2, "slot",     10.0,  60.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            (   3, "slot",     10.0,  90.0,   8.0, 40.0, 90.0, 2.0, None,  None),
            (   4, "hole",     30.0,  60.0,  12.0, 12.0,  0.0, 2.0,  6.0,  None),
            (   5, "hole",    120.0,  75.0,  30.0, 30.0,  0.0, 2.0, 15.0,  None),
            (   6, "cutout",   80.0,  40.0,  35.0, 25.0,  0.0, 2.0, None,  None),
            (   7, "cutout",  130.0,  90.0,  25.0, 25.0, 45.0, 2.0, None,  None),
            (   8, "special",  90.0,  30.0,  20.0, 20.0,  0.0, 2.0, None,  "ELEC-OUTLET"),
            (   9, "irregular",140.0, 110.0,  25.0, 15.0, 30.0, 2.0, None,  None),
        ],
    },
}

print(f"מסד נתונים נטען: {len(_V1_DATABASE)} מודל(ים)")
print(f"  PART-001: {len(_V1_DATABASE['PART-001']['features'])} features")
print(f"  רשומה גולמית לדוגמה: {_V1_DATABASE['PART-001']['features'][0]}")
print(f"  קריאה עם קבועים: FID={_V1_DATABASE['PART-001']['features'][0][FID]}, "
      f"FTYPE={_V1_DATABASE['PART-001']['features'][0][FTYPE]!r}, "
      f"X={_V1_DATABASE['PART-001']['features'][0][X]}")

מסד נתונים נטען: 1 מודל(ים)
  PART-001: 9 features
  רשומה גולמית לדוגמה: (1, 'slot', 10.0, 30.0, 40.0, 8.0, 0.0, 2.0, None, None)
  קריאה עם קבועים: FID=1, FTYPE='slot', X=10.0


---
## חלק B: Trace + מונה קריאות (כלי למידה)
אלה **לא** חלק מ-V1. הוספנו אותם כדי **לראות את רעש ה-API** שהעיצוב של V1 כופה עלינו.  
ה-trace מדפיס כל קריאה משמעותית; המונה סוכם אותן בסוף.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק B: Trace + מונה קריאות
# ────────────────────────────────────────────────────────────

TRACE_ENABLED: bool = True    # שנה ל-False כדי להשתיק
CALL_COUNTS: dict[str, int] = {}


def trace(tag: str, message: str) -> None:
    """מדפיס שורת trace מתויגת אם המעקב דלוק."""
    if TRACE_ENABLED:
        print(f"  [{tag}] {message}")


def count_call(func_name: str) -> None:
    """מגדיל את מונה הקריאות לפונקציית API של V1."""
    CALL_COUNTS[func_name] = CALL_COUNTS.get(func_name, 0) + 1


def reset_counters() -> None:
    """מאפס את כל המונים (קרא לפני כל הרצת דמו)."""
    CALL_COUNTS.clear()


def print_call_counts() -> None:
    """מציג כמה קריאות API בוצעו, ממוין לפי כמות."""
    total = sum(CALL_COUNTS.values())
    unique = len(CALL_COUNTS)
    print(f"\n{'─'*55}")
    print(f"  מונה קריאות V1 API  (סה\"כ: {total} קריאות, {unique} פונקציות שונות)")
    print(f"{'─'*55}")
    for name, count in sorted(CALL_COUNTS.items(), key=lambda x: -x[1]):
        bar = '█' * count
        print(f"  {name:<35} {count:>3}  {bar}")
    print(f"{'─'*55}")
    print(f"  סה\"כ: {total} קריאות API ב-{unique} פונקציות שונות")
    print(f"  ← זהו 'רעש ה-API' ש-V1 כופה על כל צרכן.")


print("Trace + מונה קריאות מוכנים.")
print(f"  TRACE_ENABLED = {TRACE_ENABLED}")

Trace + מונה קריאות מוכנים.
  TRACE_ENABLED = True


---
## חלק C: ניהול handles
V1 משתמש ב**handles — מספרים שלמים אטומים** — כמו file descriptors ב-C.  
הקורא חייב לעשות `open` לפני שימוש ו-`close` אחרי. שכחת `close` = דליפת משאבים.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק C: ניהול handles
# ────────────────────────────────────────────────────────────

_open_handles: dict[int, str] = {}   # handle ← שם_מודל
_next_handle: int = 0


def v1_open_model(model_name: str) -> int:
    """פותח מודל, מחזיר handle — מספר שלם אטום."""
    global _next_handle
    if model_name not in _V1_DATABASE:
        raise ValueError(f"V1: מודל {model_name!r} לא נמצא")
    _next_handle += 1
    _open_handles[_next_handle] = model_name
    trace("OPEN", f"model={model_name!r} → handle={_next_handle}")
    return _next_handle


def v1_close_model(handle: int) -> None:
    """משחרר את ה-handle. שכחת קריאה = דליפת משאבים."""
    trace("CLOSE", f"handle={handle}")
    _open_handles.pop(handle, None)


def _get_model_by_handle(handle: int) -> dict:
    """פנימי: ממיר handle לנתוני המודל."""
    return _V1_DATABASE[_open_handles[handle]]


def _get_feature_record(handle: int, feature_id: int) -> tuple:
    """פנימי: סריקה ליניארית למציאת feature לפי ID."""
    for record in _get_model_by_handle(handle)["features"]:
        if record[FID] == feature_id:
            return record
    raise ValueError(f"V1: feature {feature_id} לא נמצא")


# בדיקה מהירה: פתיחה וסגירה
h = v1_open_model("PART-001")
print(f"  סוג handle: {type(h).__name__}, ערך: {h}")
v1_close_model(h)
print(f"  handles פתוחים אחרי סגירה: {len(_open_handles)}")

  [OPEN] model='PART-001' → handle=1
  סוג handle: int, ערך: 1
  [CLOSE] handle=1
  handles פתוחים אחרי סגירה: 0


---
## חלק D: API ברמת המודל
הפונקציות האלה שואלות על **המודל כמכלול** — כמה features, מידות היריעה, חומר.  
הן זהות בלי קשר לסוג ה-feature. הכאב מתחיל ב**חלק הבא**.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק D: API ברמת המודל
# ────────────────────────────────────────────────────────────

def v1_get_num_features(handle: int) -> int:
    """מחזיר את מספר ה-features במודל."""
    count_call("v1_get_num_features")
    count = len(_get_model_by_handle(handle)["features"])
    trace("COUNT", f"features={count}")
    return count


def v1_get_feature_id(handle: int, index: int) -> int:
    """מחזיר את ה-ID של feature באינדקס נתון (מבוסס 0)."""
    count_call("v1_get_feature_id")
    return _get_model_by_handle(handle)["features"][index][FID]


def v1_get_feature_type(handle: int, index: int) -> str:
    """מחזיר את מחרוזת הסוג באינדקס נתון."""
    count_call("v1_get_feature_type")
    return _get_model_by_handle(handle)["features"][index][FTYPE]


def v1_get_sheet_width(handle: int) -> float:
    """מחזיר את רוחב יריעת המתכת."""
    count_call("v1_get_sheet_width")
    return _get_model_by_handle(handle)["sheet_width"]


def v1_get_sheet_height(handle: int) -> float:
    """מחזיר את גובה יריעת המתכת."""
    count_call("v1_get_sheet_height")
    return _get_model_by_handle(handle)["sheet_height"]


def v1_get_material(handle: int) -> str:
    """מחזיר את סוג החומר."""
    count_call("v1_get_material")
    return _get_model_by_handle(handle)["material"]


print("API ברמת המודל מוכן: v1_get_num_features, v1_get_feature_id, v1_get_feature_type, ...")

API ברמת המודל מוכן: v1_get_num_features, v1_get_feature_id, v1_get_feature_type, ...


---
## חלק E: Getters תלויי סוג (החלק הכואב)
ב-V1 האמיתי, אלה היו **פונקציות C שונות** ב**ספריות שונות**.  
קריאה לפונקציה הלא-נכונה עבור סוג נתון = קריסה.

שימו לב: `v1_get_x_for_slot` ו-`v1_get_x_for_hole` שתיהן מחזירות `record[X]` — **אותו מידע בדיוק!**  
אבל V1 מכריח אותנו לקרוא לפונקציות שונות לפי סוג. זה הכאב המרכזי.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק E: Getters תלויי סוג (החלק הכואב)
# ────────────────────────────────────────────────────────────

# -- Getters של Slot (6 פונקציות) --
def v1_get_x_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_x_for_slot")
    return _get_feature_record(h, fid)[X]

def v1_get_y_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_y_for_slot")
    return _get_feature_record(h, fid)[Y]

def v1_get_w_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_w_for_slot")
    return _get_feature_record(h, fid)[WIDTH]

def v1_get_h_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_h_for_slot")
    return _get_feature_record(h, fid)[HEIGHT]

def v1_get_angle_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_angle_for_slot")
    return _get_feature_record(h, fid)[ANGLE]

def v1_get_depth_for_slot(h: int, fid: int) -> float:
    count_call("v1_get_depth_for_slot")
    return _get_feature_record(h, fid)[DEPTH]


# -- Getters של Hole (4 פונקציות) --
def v1_get_x_for_hole(h: int, fid: int) -> float:
    count_call("v1_get_x_for_hole")
    return _get_feature_record(h, fid)[X]

def v1_get_y_for_hole(h: int, fid: int) -> float:
    count_call("v1_get_y_for_hole")
    return _get_feature_record(h, fid)[Y]

def v1_get_radius_for_hole(h: int, fid: int) -> float:
    count_call("v1_get_radius_for_hole")
    return _get_feature_record(h, fid)[RADIUS]

def v1_get_depth_for_hole(h: int, fid: int) -> float:
    count_call("v1_get_depth_for_hole")
    return _get_feature_record(h, fid)[DEPTH]


# -- Getters של Cutout (5 פונקציות) --
def v1_get_x_for_cutout(h: int, fid: int) -> float:
    count_call("v1_get_x_for_cutout")
    return _get_feature_record(h, fid)[X]

def v1_get_y_for_cutout(h: int, fid: int) -> float:
    count_call("v1_get_y_for_cutout")
    return _get_feature_record(h, fid)[Y]

def v1_get_w_for_cutout(h: int, fid: int) -> float:
    count_call("v1_get_w_for_cutout")
    return _get_feature_record(h, fid)[WIDTH]

def v1_get_h_for_cutout(h: int, fid: int) -> float:
    count_call("v1_get_h_for_cutout")
    return _get_feature_record(h, fid)[HEIGHT]

def v1_get_angle_for_cutout(h: int, fid: int) -> float:
    count_call("v1_get_angle_for_cutout")
    return _get_feature_record(h, fid)[ANGLE]


# -- Getters של Special (3 פונקציות) --
def v1_get_x_for_special(h: int, fid: int) -> float:
    count_call("v1_get_x_for_special")
    return _get_feature_record(h, fid)[X]

def v1_get_y_for_special(h: int, fid: int) -> float:
    count_call("v1_get_y_for_special")
    return _get_feature_record(h, fid)[Y]

def v1_get_code_for_special(h: int, fid: int) -> str:
    count_call("v1_get_code_for_special")
    return _get_feature_record(h, fid)[SPECIAL_CODE]


# -- Getters של Irregular (4 פונקציות) --
def v1_get_x_for_irregular(h: int, fid: int) -> float:
    count_call("v1_get_x_for_irregular")
    return _get_feature_record(h, fid)[X]

def v1_get_y_for_irregular(h: int, fid: int) -> float:
    count_call("v1_get_y_for_irregular")
    return _get_feature_record(h, fid)[Y]

def v1_get_bbox_w_for_irregular(h: int, fid: int) -> float:
    count_call("v1_get_bbox_w_for_irregular")
    return _get_feature_record(h, fid)[WIDTH]

def v1_get_bbox_h_for_irregular(h: int, fid: int) -> float:
    count_call("v1_get_bbox_h_for_irregular")
    return _get_feature_record(h, fid)[HEIGHT]


print("Getters תלויי סוג מוכנים:")
print("  Slot:      v1_get_x_for_slot, v1_get_y_for_slot, ... (6 פונקציות)")
print("  Hole:      v1_get_x_for_hole, v1_get_radius_for_hole, ... (4 פונקציות)")
print("  Cutout:    v1_get_x_for_cutout, ... (5 פונקציות)")
print("  Special:   v1_get_x_for_special, v1_get_code_for_special, ... (3 פונקציות)")
print("  Irregular: v1_get_x_for_irregular, v1_get_bbox_w_for_irregular, ... (4 פונקציות)")
print(f"  סה\"כ: 22 פונקציות getter תלויות סוג")

Getters תלויי סוג מוכנים:
  Slot:      v1_get_x_for_slot, v1_get_y_for_slot, ... (6 פונקציות)
  Hole:      v1_get_x_for_hole, v1_get_radius_for_hole, ... (4 פונקציות)
  Cutout:    v1_get_x_for_cutout, ... (5 פונקציות)
  Special:   v1_get_x_for_special, v1_get_code_for_special, ... (3 פונקציות)
  Irregular: v1_get_x_for_irregular, v1_get_bbox_w_for_irregular, ... (4 פונקציות)
  סה"כ: 22 פונקציות getter תלויות סוג


---
## חלק F: פונקציות חילוץ (אחת לכל סוג feature)
כל פונקציה קוראת ל-getters הנכונים של V1 עבור הסוג שלה ומחזירה dict  
שמערכת המומחה יכולה לצרוך.

ה-`if/elif` בפונקציה הראשית (חלק הבא) נשאר — אבל עכשיו כל ענף  
מאציל לפונקציית עזר קטנה וממוקדת. קל להשוות מה `slot` צריך מול `hole`.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק F: פונקציות חילוץ (אחת לכל סוג)
# ────────────────────────────────────────────────────────────

def extract_slot(handle: int, feature_id: int) -> dict:
    """מחלץ feature מסוג slot דרך getters ייעודיים של V1."""
    return {
        "id":     feature_id,
        "type":   "slot",
        "x":      v1_get_x_for_slot(handle, feature_id),
        "y":      v1_get_y_for_slot(handle, feature_id),
        "width":  v1_get_w_for_slot(handle, feature_id),
        "height": v1_get_h_for_slot(handle, feature_id),
        "angle":  v1_get_angle_for_slot(handle, feature_id),
        "depth":  v1_get_depth_for_slot(handle, feature_id),
    }


def extract_hole(handle: int, feature_id: int) -> dict:
    """מחלץ feature מסוג hole דרך getters ייעודיים של V1."""
    return {
        "id":     feature_id,
        "type":   "hole",
        "x":      v1_get_x_for_hole(handle, feature_id),
        "y":      v1_get_y_for_hole(handle, feature_id),
        "radius": v1_get_radius_for_hole(handle, feature_id),
        "depth":  v1_get_depth_for_hole(handle, feature_id),
    }


def extract_cutout(handle: int, feature_id: int) -> dict:
    """מחלץ feature מסוג cutout דרך getters ייעודיים של V1."""
    return {
        "id":     feature_id,
        "type":   "cutout",
        "x":      v1_get_x_for_cutout(handle, feature_id),
        "y":      v1_get_y_for_cutout(handle, feature_id),
        "width":  v1_get_w_for_cutout(handle, feature_id),
        "height": v1_get_h_for_cutout(handle, feature_id),
        "angle":  v1_get_angle_for_cutout(handle, feature_id),
    }


def extract_special(handle: int, feature_id: int) -> dict:
    """מחלץ feature מסוג special דרך getters ייעודיים של V1."""
    return {
        "id":   feature_id,
        "type": "special",
        "x":    v1_get_x_for_special(handle, feature_id),
        "y":    v1_get_y_for_special(handle, feature_id),
        "code": v1_get_code_for_special(handle, feature_id),
    }


def extract_irregular(handle: int, feature_id: int) -> dict:
    """מחלץ feature מסוג irregular דרך getters ייעודיים של V1."""
    return {
        "id":     feature_id,
        "type":   "irregular",
        "x":      v1_get_x_for_irregular(handle, feature_id),
        "y":      v1_get_y_for_irregular(handle, feature_id),
        "bbox_w": v1_get_bbox_w_for_irregular(handle, feature_id),
        "bbox_h": v1_get_bbox_h_for_irregular(handle, feature_id),
    }


print("פונקציות חילוץ מוכנות: extract_slot, extract_hole, extract_cutout, extract_special, extract_irregular")

פונקציות חילוץ מוכנות: extract_slot, extract_hole, extract_cutout, extract_special, extract_irregular


---
## חלק G: הפונקציה הראשית — תזמורן החילוץ (עם טבלת ניתוב)
במקום `if/elif` ענק, אנחנו משתמשים ב**טבלת ניתוב (dispatch table)** —  
dict שממפה מחרוזת סוג לפונקציה שמטפלת בו.

זה עדיין **פרוצדורלי לחלוטין** (dict של פונקציות הוא לא OOP),  
אבל מנקה את ה-`if/elif` ומאפשר הוספת סוג חדש בשורה אחת בלבד.

**הכאב לא נעלם** — עדיין יש getters תלויי-סוג, ניהול ידני של handle,  
ושכפול dispatch אצל כל צרכן. אבל עכשיו הטבלה לפחות מרכזת את המיפוי במקום אחד.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק G: הפונקציה הראשית — תזמורן החילוץ (עם טבלת ניתוב)
# ────────────────────────────────────────────────────────────

# טבלת הניתוב: מיפוי בין סוג ה-feature לפונקציה המטפלת בו
FEATURE_EXTRACTORS: dict[str, callable] = {
    "slot":      extract_slot,
    "hole":      extract_hole,
    "cutout":    extract_cutout,
    "special":   extract_special,
    "irregular": extract_irregular,
}


def extract_features_v1(model_name: str) -> list[dict]:
    """
    פותחת מודל דרך V1 API, מחלצת כל feature
    ל-dict שטוח שמערכת המומחה יכולה לצרוך.
    """
    handle = v1_open_model(model_name)
    try:
        num_features = v1_get_num_features(handle)
        results: list[dict] = []

        for index in range(num_features):
            feature_id   = v1_get_feature_id(handle, index)
            feature_type = v1_get_feature_type(handle, index)
            trace("FEATURE", f"index={index} id={feature_id} type={feature_type!r}")

            # ---- שימוש בטבלת הניתוב במקום if/elif ----
            extractor_func = FEATURE_EXTRACTORS.get(feature_type)

            if extractor_func is None:
                raise ValueError(f"סוג feature לא מוכר: {feature_type!r}")

            # קריאה לפונקציה ושמירת התוצאה
            results.append(extractor_func(handle, feature_id))

        return results

    finally:
        v1_close_model(handle)   # אסור לשכוח!


print("תזמורן החילוץ מוכן: extract_features_v1() — עכשיו עם טבלת ניתוב")

תזמורן החילוץ מוכן: extract_features_v1() — עכשיו עם טבלת ניתוב


---
## חלק H: מערכת המומחה (stub)
מערכת המומחה האמיתית גדולה ויקרה. אנחנו רק מוכיחים שהמידע זורם נכון.

**שימו לב:** למערכת המומחה יש **if/elif משלה על הסוג** — לוגיקת ה-dispatch  
**משוכפלת** לאורך כל הקוד. כל צרכן חדש חוזר על אותה תבנית.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק H: מערכת המומחה (stub)
# ────────────────────────────────────────────────────────────

def expert_system_process(features: list[dict]) -> list[str]:
    """צורכת dicts של features, מחזירה פקודות NC (מפושט)."""
    commands: list[str] = []
    for feat in features:
        t = feat["type"]
        # --- עוד if/elif על הסוג! שכפול! ---
        if t == "slot":
            commands.append(
                f"NC: CUT_SLOT at ({feat['x']},{feat['y']}) "
                f"w={feat['width']} h={feat['height']} "
                f"angle={feat['angle']} depth={feat['depth']}"
            )
        elif t == "hole":
            commands.append(
                f"NC: DRILL_HOLE at ({feat['x']},{feat['y']}) "
                f"r={feat['radius']} depth={feat['depth']}"
            )
        elif t == "cutout":
            commands.append(
                f"NC: PUNCH_CUTOUT at ({feat['x']},{feat['y']}) "
                f"w={feat['width']} h={feat['height']} angle={feat['angle']}"
            )
        elif t == "special":
            commands.append(
                f"NC: PUNCH_SPECIAL at ({feat['x']},{feat['y']}) "
                f"code={feat['code']}"
            )
        elif t == "irregular":
            commands.append(
                f"NC: MULTI_CUT at ({feat['x']},{feat['y']}) "
                f"bbox={feat['bbox_w']}x{feat['bbox_h']}"
            )
    return commands


print("מערכת המומחה (stub) מוכנה: expert_system_process()")

מערכת המומחה (stub) מוכנה: expert_system_process()


---
## חלק I: הרצה מודרכת — הסבר על feature אחד
במקום לראות את כל 9 ה-features בבת אחת, עוברים על **feature בודד** לאט:  
**tuple גולמי → החלטת dispatch → dict מעובד**.  
זו הדרך הכי ברורה לראות את זרימת החילוץ של V1.

In [ ]:
# ────────────────────────────────────────────────────────────
# חלק I: הרצה מודרכת — הסבר על feature אחד
# ────────────────────────────────────────────────────────────

def explain_feature_extraction(model_name: str, index: int) -> None:
    """
    חילוץ צעד-אחר-צעד של feature אחד באינדקס הנתון.
    מציגה את הרשומה הגולמית, את החלטת הניתוב, ואת התוצאה.
    """
    print(f"\n{'═'*55}")
    print(f"  הרצה מודרכת: feature באינדקס {index}")
    print(f"{'═'*55}")

    handle = v1_open_model(model_name)
    try:
        feature_id   = v1_get_feature_id(handle, index)
        feature_type = v1_get_feature_type(handle, index)

        # שלב 1: הצגת ה-tuple הגולמי (מה ש-V1 באמת שומר)
        raw_record = _get_feature_record(handle, feature_id)
        print(f"\n  שלב 1 — רשומה גולמית (tuple ממסד הנתונים של V1):")
        print(f"    {raw_record}")
        print(f"    שדות: FID={raw_record[FID]}, FTYPE={raw_record[FTYPE]!r}, "
              f"X={raw_record[X]}, Y={raw_record[Y]}, "
              f"W={raw_record[WIDTH]}, H={raw_record[HEIGHT]}")

        # שלב 2: הצגת החלטת הניתוב
        print(f"\n  שלב 2 — ניתוב לפי סוג:")
        print(f"    feature_type = {feature_type!r}")
        print(f"    → FEATURE_EXTRACTORS[{feature_type!r}] = {FEATURE_EXTRACTORS[feature_type].__name__}()")
        print(f"    → חייבים לקרוא לפונקציות v1_get_*_for_{feature_type}()")

        # שלב 3: הצגת ה-dict המעובד (דרך טבלת הניתוב)
        extractor_func = FEATURE_EXTRACTORS.get(feature_type)
        processed = extractor_func(handle, feature_id) if extractor_func else {}

        print(f"\n  שלב 3 — dict מעובד (מה שמערכת המומחה מקבלת):")
        for key, value in processed.items():
            print(f"    {key:<8} = {value}")

        print(f"\n{'═'*55}")

    finally:
        v1_close_model(handle)


print("הרצה מודרכת מוכנה: explain_feature_extraction()")

הרצה מודרכת מוכנה: explain_feature_extraction()


---
## דמו 1: חילוץ מלא עם trace
מריצים את כל הצינור: פתיחת מודל → לולאת features → dispatch לפי סוג → סגירת מודל.  
עקבו אחרי תגיות ה-trace בפלט: `[OPEN]`, `[COUNT]`, `[FEATURE]`, `[CLOSE]`.

In [ ]:
# ════════════════════════════════════════════════════════════
# דמו 1: חילוץ מלא עם trace
# ════════════════════════════════════════════════════════════

TRACE_ENABLED = True
reset_counters()

print("═" * 60)
print("  דמו 1: חילוץ V1 מלא (עם trace)")
print("═" * 60)

features = extract_features_v1("PART-001")

print(f"\nחולצו {len(features)} features:\n")
for f in features:
    print(f"  #{f['id']:>2}  {f['type']:<10}  {f}")

════════════════════════════════════════════════════════════
  דמו 1: חילוץ V1 מלא (עם trace)
════════════════════════════════════════════════════════════
  [OPEN] model='PART-001' → handle=2
  [COUNT] features=9
  [FEATURE] index=0 id=1 type='slot'
  [FEATURE] index=1 id=2 type='slot'
  [FEATURE] index=2 id=3 type='slot'
  [FEATURE] index=3 id=4 type='hole'
  [FEATURE] index=4 id=5 type='hole'
  [FEATURE] index=5 id=6 type='cutout'
  [FEATURE] index=6 id=7 type='cutout'
  [FEATURE] index=7 id=8 type='special'
  [FEATURE] index=8 id=9 type='irregular'
  [CLOSE] handle=2

חולצו 9 features:

  # 1  slot        {'id': 1, 'type': 'slot', 'x': 10.0, 'y': 30.0, 'width': 40.0, 'height': 8.0, 'angle': 0.0, 'depth': 2.0}
  # 2  slot        {'id': 2, 'type': 'slot', 'x': 10.0, 'y': 60.0, 'width': 40.0, 'height': 8.0, 'angle': 0.0, 'depth': 2.0}
  # 3  slot        {'id': 3, 'type': 'slot', 'x': 10.0, 'y': 90.0, 'width': 8.0, 'height': 40.0, 'angle': 90.0, 'depth': 2.0}
  # 4  hole        {'id': 4

---
## דמו 2: פקודות NC של מערכת המומחה
מזינים את ה-dicts שחולצו למערכת המומחה ורואים את פלט ה-NC.

In [ ]:
# ════════════════════════════════════════════════════════════
# דמו 2: פקודות NC של מערכת המומחה
# ════════════════════════════════════════════════════════════

print("─" * 60)
print("  דמו 2: פקודות NC של מערכת המומחה")
print("─" * 60)
print()

nc_commands = expert_system_process(features)
for cmd in nc_commands:
    print(f"  {cmd}")

────────────────────────────────────────────────────────────
  דמו 2: פקודות NC של מערכת המומחה
────────────────────────────────────────────────────────────

  NC: CUT_SLOT at (10.0,30.0) w=40.0 h=8.0 angle=0.0 depth=2.0
  NC: CUT_SLOT at (10.0,60.0) w=40.0 h=8.0 angle=0.0 depth=2.0
  NC: CUT_SLOT at (10.0,90.0) w=8.0 h=40.0 angle=90.0 depth=2.0
  NC: DRILL_HOLE at (30.0,60.0) r=6.0 depth=2.0
  NC: DRILL_HOLE at (120.0,75.0) r=15.0 depth=2.0
  NC: PUNCH_CUTOUT at (80.0,40.0) w=35.0 h=25.0 angle=0.0
  NC: PUNCH_CUTOUT at (130.0,90.0) w=25.0 h=25.0 angle=45.0
  NC: PUNCH_SPECIAL at (90.0,30.0) code=ELEC-OUTLET
  NC: MULTI_CUT at (140.0,110.0) bbox=25.0x15.0


---
## דמו 3: רעש API — מונה קריאות
כמה קריאות V1 API נדרשו כדי לחלץ 9 features?  
תרשים העמודות הופך את **הרעש** של העיצוב הפרוצדורלי של V1 לנראה.

In [ ]:
# ════════════════════════════════════════════════════════════
# דמו 3: רעש API — מונה קריאות
# ════════════════════════════════════════════════════════════

print_call_counts()


───────────────────────────────────────────────────────
  מונה קריאות V1 API  (סה"כ: 62 קריאות, 25 פונקציות שונות)
───────────────────────────────────────────────────────
  v1_get_feature_id                     9  █████████
  v1_get_feature_type                   9  █████████
  v1_get_x_for_slot                     3  ███
  v1_get_y_for_slot                     3  ███
  v1_get_w_for_slot                     3  ███
  v1_get_h_for_slot                     3  ███
  v1_get_angle_for_slot                 3  ███
  v1_get_depth_for_slot                 3  ███
  v1_get_x_for_hole                     2  ██
  v1_get_y_for_hole                     2  ██
  v1_get_radius_for_hole                2  ██
  v1_get_depth_for_hole                 2  ██
  v1_get_x_for_cutout                   2  ██
  v1_get_y_for_cutout                   2  ██
  v1_get_w_for_cutout                   2  ██
  v1_get_h_for_cutout                   2  ██
  v1_get_angle_for_cutout               2  ██
  v1_get_num_features     

---
## דמו 4: הרצה מודרכת — slot אחד ו-hole אחד
רואים את שלושת השלבים (גולמי → dispatch → מעובד) ל**feature בודד** בכל פעם.  
ה-trace כבוי כאן כדי שהפלט יהיה נקי.

In [ ]:
# ════════════════════════════════════════════════════════════
# דמו 4: הרצה מודרכת — slot אחד ו-hole אחד
# ════════════════════════════════════════════════════════════

TRACE_ENABLED = True   # פלט נקי להרצה מודרכת

explain_feature_extraction("PART-001", index=0)   # slot
explain_feature_extraction("PART-001", index=3)   # hole

TRACE_ENABLED = True    # משחזרים לתאים הבאים


═══════════════════════════════════════════════════════
  הרצה מודרכת: feature באינדקס 0
═══════════════════════════════════════════════════════
  [OPEN] model='PART-001' → handle=3

  שלב 1 — רשומה גולמית (tuple ממסד הנתונים של V1):
    (1, 'slot', 10.0, 30.0, 40.0, 8.0, 0.0, 2.0, None, None)
    שדות: FID=1, FTYPE='slot', X=10.0, Y=30.0, W=40.0, H=8.0

  שלב 2 — ניתוב לפי סוג:
    feature_type = 'slot'
    → FEATURE_EXTRACTORS['slot'] = extract_slot()
    → חייבים לקרוא לפונקציות v1_get_*_for_slot()

  שלב 3 — dict מעובד (מה שמערכת המומחה מקבלת):
    id       = 1
    type     = slot
    x        = 10.0
    y        = 30.0
    width    = 40.0
    height   = 8.0
    angle    = 0.0
    depth    = 2.0

═══════════════════════════════════════════════════════
  [CLOSE] handle=3

═══════════════════════════════════════════════════════
  הרצה מודרכת: feature באינדקס 3
═══════════════════════════════════════════════════════
  [OPEN] model='PART-001' → handle=4

  שלב 1 — רשומה גולמית (tuple 

---
## בדיקות (assertions)

In [ ]:
# ════════════════════════════════════════════════════════════
# בדיקות
# ════════════════════════════════════════════════════════════

# מספר נכון של features
assert len(features) == 9, f"צפוי 9, קיבלנו {len(features)}"

# נתוני slot נכונים
assert features[0]["type"] == "slot"
assert features[0]["x"] == 10.0
assert features[0]["width"] == 40.0

# ל-hole יש radius, לא width/height
assert "radius" in features[3]
assert features[3]["radius"] == 6.0
assert "width" not in features[3]   # ל-holes אין מפתח width

# ל-special יש code
assert features[7]["code"] == "ELEC-OUTLET"

# פקודות NC מתאימות 1:1
assert len(nc_commands) == 9
assert "CUT_SLOT" in nc_commands[0]
assert "DRILL_HOLE" in nc_commands[3]
assert "PUNCH_SPECIAL" in nc_commands[7]
assert "MULTI_CUT" in nc_commands[8]

# ה-handle שוחרר
assert len(_open_handles) == 0, "דליפת handle!"

# מונה הקריאות מוכיח רעש API
assert sum(CALL_COUNTS.values()) > 30, "ציפינו להרבה קריאות V1 API"

print("כל הבדיקות עברו בהצלחה.")

כל הבדיקות עברו בהצלחה.


---

## לקחים — מה כואב בעיצוב הזה

### 1. ניהול ידני של handles
הקורא חייב לזכור לקרוא ל-`v1_close_model`. שכחת זה = דליפת משאבים. כל פונקציה דורשת שה-handle יועבר אליה — למערכת אין מושג של "המודל הזה" כאובייקט.

### 2. if/elif ענק על הסוג
ל-`extract_features_v1` יש **5 ענפי if/elif** שמנתבים לפונקציות שונות לפי סוג ה-feature. אם מוסיפים סוג שישי, חייבים להוסיף ענף נוסף — ו**כל צרכן** של V1 חייב לעשות את אותו דבר.

### 3. שמות פונקציות תלויי סוג
ל-V1 יש `v1_get_x_for_slot`, `v1_get_x_for_hole`, `v1_get_x_for_cutout` — כולן עושות אותו דבר (מחזירות x) אבל לסוגים שונים. **מונה הקריאות** מוכיח את זה: חילוץ 9 features דורש 62 קריאות API ב-25 פונקציות שונות. זהו "רעש ה-API" ש-V1 כופה על כל צרכן.

### 4. גם מערכת המומחה צריכה dispatch לפי סוג
ל-`expert_system_process` יש **שרשרת if/elif משלה**. לוגיקת ה-dispatch משוכפלת לאורך כל הקוד. כל צרכן חדש חוזר על אותה תבנית.

### 5. אין בידוד משינויים
כש-V2 יגיע (מונחה עצמים, API שונה, שמות שונים), נצטרך לכתוב פונקציית חילוץ **נפרדת לחלוטין**. וגם V3 אחרי זה. מערכת המומחה צמודה חזק לגרסה שממנה היא מקבלת נתונים.

---

### למה שלב 2 הופך להכרחי

V2 מונחה עצמים — הוא מחזיר אובייקטים אמיתיים (`OOGFeature`) במקום לדרוש handle + ID + dispatch לפי סוג. זה נשמע כמו התקדמות! אבל הממשק של V2 משתמש ב**שמות שונים** (למשל `get_length()` איפה שאנחנו מצפים ל-`get_width()`), ו**חסרים** לו דברים שאנחנו צריכים (כמו `generate_nc_commands()`).

אז בשלב 2 נעשה:
- נציג את אובייקטי ה-OOG של V2.
- ננסה גישת עטיפה OO סבירה.
- נגלה את **בעיית פיצוץ המחלקות**: 5 סוגי features × N גרסאות = יותר מדי מחלקות.

הכאב הזה יניע את שלב 3, שבו patterns (Bridge, Adapter, Facade) נותנים לנו ארכיטקטורה נקייה.